This notebook ultilize the gaussian_filter to make the FE smooth

In [ ]:
# --- Run provenance: start of run -------------------------------------
# Records when this notebook run started and where it ran. The last cell
# of the notebook records the stop time, the notebook's settings, software
# versions and checksums. Keep this cell first and run it before the
# analysis cells.
import os
import time

_prov_start = time.time()
print("provenance start:", time.strftime("%Y-%m-%dT%H:%M:%S"))
print("working directory:", os.getcwd())

In [ ]:
import deeptime as dpt
from tqdm.notebook import tqdm # for progress bar
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pyemma as pem
import pickle
import os
from scipy.ndimage import gaussian_filter
from matplotlib.colors import PowerNorm

In [ ]:
# Step 1: Load your Q-G data
data = np.load("QG.npy")  # shape (50, 13333, 2)

"""
In general, I want to do MSM on QG space. However, some proteins do not sample the entangled state.
Hence, in that case we need to do MSM on Q space alone.
By defaults, n_dims is 2- mean perform on QG space. If not working, then set to n_dims=1 to use Q only
"""
# Set the number of dimensions to use for the MSM
n_dims = 2

if n_dims == 2: 
    # Step 2: Standardize the data
    scaler = StandardScaler()
    raw_data_all = np.concatenate(data, axis=0)
    scaler.fit(raw_data_all)
    processed_data = [scaler.transform(traj) for traj in data]  # List of 2D arrays
    conc_processed_data = np.concatenate(processed_data, axis=0)

elif n_dims == 1:
    # Incase MSM is not working on GQ space since the G is not significant, did MSM on Q space alone
    # Step 2: Load your Q-G data
    data = data[:, :, 0][:, :, np.newaxis]  # shape becomes (50, 13333, 1)
    # Step 2: Standardize the data
    scaler = StandardScaler()
    raw_data_all = np.concatenate(data, axis=0)
    scaler.fit(raw_data_all)
    processed_data = [scaler.transform(traj) for traj in data]  # List of 2D arrays
    conc_processed_data = np.concatenate(processed_data, axis=0)

In [ ]:
Q = raw_data_all[:,0]
G = raw_data_all[:,1]

In [ ]:
# Set number of bins for histogram
nbins = [100, 50]

# Step 1: Compute 2D histogram
H, xedges, yedges = np.histogram2d(Q, G, bins=nbins)
H = H.T  # Transpose for correct orientation (Y, X)

# Step 2: Normalize to get probability density
P = H / np.sum(H)

# Step 3: Create mask for unsampled bins
mask = P == 0

# Step 4: Convert to free energy (in kBT units)
F = -np.log(P + 1e-10)  # small value to avoid log(0)
F -= np.min(F)          # shift minimum to zero

# Step 5: Smooth the free energy surface
F_smooth = gaussian_filter(F, sigma=[1, 5])

# Step 6: Apply mask to unsampled regions
F_smooth_masked = np.ma.masked_where(mask, F_smooth)

# Optional: clip high energies for better visualization (like PyEMMA)
# F_plot = np.ma.clip(F_smooth_masked, 0, 12)
F_plot = F_smooth_masked  # No clipping
F_plot -= np.min(F_plot)
# Step 7: Create meshgrid for plotting
xcenters = 0.5 * (xedges[:-1] + xedges[1:])
ycenters = 0.5 * (yedges[:-1] + yedges[1:])
X, Y = np.meshgrid(xcenters, ycenters)

# Step 8: Define colormap (white for masked regions)
cmap = plt.cm.coolwarm
# cmap = plt.cm.plasma
cmap.set_bad(color='white')

# Step 9: Plot the free energy landscape
plt.figure(figsize=(6, 5))
# cf = plt.contourf(X, Y, F_plot, levels=50, cmap=cmap)
# Use a power-law normalization (e.g. gamma < 1 boosts contrast in low values)
norm = PowerNorm(gamma=0.5, vmin=np.min(F_plot), vmax=np.max(F_plot))

# Then plot with this norm:
cf = plt.contourf(X, Y, F_plot, levels=20, cmap=cmap, norm=norm)

plt.colorbar(cf, label='Free Energy (kBT)')
plt.xlabel('Q')
plt.ylabel('G')
plt.title('Smoothed Free Energy Landscape with Masked Unsampled Regions')
plt.tight_layout()
plt.show()

In [ ]:
# --- Run provenance: end of run ---------------------------------------
# Prints the stop time, the values of the notebook's own variables (its
# settings), interpreter and package versions, and sha256 checksums of the
# files whose paths were written in the executed cells and of the files
# this run wrote. Keep this cell last.
import os
import re
import time
import types
import hashlib
import platform
from importlib.metadata import distributions


def _prov_sha256(path):
    _h = hashlib.sha256()
    with open(path, "rb") as _fh:
        for _chunk in iter(lambda: _fh.read(1 << 20), b""):
            _h.update(_chunk)
    return _h.hexdigest()


_prov_t0 = globals().get("_prov_start", time.time())
print(
    "provenance end:",
    time.strftime("%Y-%m-%dT%H:%M:%S"),
    f"(wall {time.time() - _prov_t0:.1f} s)",
)
try:
    print("python:", platform.python_version())
    _pkgs = sorted(
        f"{_d.metadata['Name']}=={_d.version}"
        for _d in distributions()
        if _d.metadata and _d.metadata.get("Name")
    )
    print("packages:", "; ".join(_pkgs))
except Exception as _exc:
    print("packages: unavailable", _exc)

_skip = {"In", "Out", "get_ipython", "exit", "quit", "open"}
for _k, _v in sorted(globals().items()):
    try:
        if _k.startswith("_") or _k in _skip:
            continue
        if isinstance(_v, (types.ModuleType, types.FunctionType, type)):
            continue
        _s = repr(_v)
        print(f"setting {_k} = {_s[:1000]}"
              + (" ...[truncated]" if len(_s) > 1000 else ""))
        if isinstance(_v, (str, os.PathLike)) and os.path.isfile(os.fspath(_v)):
            print(f"sha256 {_k} ({os.fspath(_v)}): {_prov_sha256(os.fspath(_v))}")
    except Exception:
        pass

try:
    _seen = set()
    for _src in (In if "In" in globals() else []):
        for _m in re.findall(
            r"['\"]([\w./\\-]+\.(?:pkl|npy|csv|tsv|dcd|psf|dat|xml|top|prm|txt|pdf|png|html))['\"]",
            _src,
        ):
            if _m in _seen:
                continue
            _seen.add(_m)
            try:
                if os.path.isfile(_m):
                    print(f"sha256 {_m}: {_prov_sha256(_m)}")
            except Exception:
                pass
except Exception:
    pass

try:
    for _n in sorted(os.listdir(".")):
        if _n.endswith(".ipynb"):
            continue
        _p = os.path.join(".", _n)
        if os.path.isfile(_p) and os.path.getmtime(_p) >= _prov_t0:
            print(f"sha256 output {_n}: {_prov_sha256(_p)}")
except Exception:
    pass